# ST456 Colab 一键运行 Notebook（ZIP 上传版）

这个 notebook 对应当前项目的新版主线：

- E1-E5 是主实验
- retrieval 只作为 appendix / 附加实验
- 不需要 GitHub
- 只需要上传一个 ZIP 压缩包，然后点击 `Runtime -> Run all`


In [ ]:
# 参数区：运行前先改这里
ZIP_EXPECTED_HINT = 'ST456group.zip'
PROJECT_SUBDIR = 'codex-novel-continuation'

USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_WORKSPACE = '/content/drive/MyDrive/st456_runs'

# 第一次建议先 False，只跑 E1 smoke。
RUN_FULL_MAINLINE = False

# retrieval 只作为 appendix。
RUN_APPENDIX_RETRIEVAL = False

# 是否自动打包并下载 artifacts 结果。
DOWNLOAD_RESULTS_ZIP = True

print('参数已加载。这个 notebook 运行时会要求你上传一个 ZIP 压缩包。')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

# 减少模型加载时的 tqdm 进度条刷屏
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

def run(cmd, cwd=None):
    """执行命令并实时显示输出，出错时抛出异常。"""
    print(f'\n>>> {cmd}', flush=True)
    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True
    )
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(
            f'命令执行失败（exit code {process.returncode}）：{cmd}\n'
            f'错误信息已打印在上方，请向上滚动查看。'
        )

workspace_root = Path('/content')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    workspace_root = Path(GOOGLE_DRIVE_WORKSPACE)

workspace_root.mkdir(parents=True, exist_ok=True)
os.chdir(workspace_root)
print('当前工作目录:', Path.cwd())

from google.colab import files
print(f'请上传项目 ZIP 压缩包，例如: {ZIP_EXPECTED_HINT}')
uploaded = files.upload()
zip_candidates = [name for name in uploaded.keys() if name.lower().endswith('.zip')]
if not zip_candidates:
    raise ValueError('没有检测到 ZIP 压缩包上传。')

zip_name = zip_candidates[0]
print('已上传 ZIP:', zip_name)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(workspace_root)

project_matches = sorted(workspace_root.rglob(PROJECT_SUBDIR))
project_matches = [path for path in project_matches if path.is_dir()]
if not project_matches:
    raise FileNotFoundError(f'解压之后没有找到 {PROJECT_SUBDIR} 目录。')

project_root = project_matches[0]
os.chdir(project_root)
print('项目目录:', Path.cwd())
run(f'{sys.executable} -m pip install -r requirements.txt')
print('环境准备完成。')

## 数据准备与 token 预算检查

这一步会：

- 下载 Sherlock Holmes 语料
- 构建数据集
- 保存 E1、E3、E5 的 token 统计结果


In [ ]:
run(f'{sys.executable} scripts/download_gutenberg.py')
run(f'{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40')

Path('artifacts/eval').mkdir(parents=True, exist_ok=True)
token_stat_configs = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml'),
]

for experiment_id, config_path in token_stat_configs:
    output_path = Path('artifacts/eval') / f'token_stats_{experiment_id.lower()}.json'
    run(f'{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}')

print('Token 统计文件:')
for path in sorted(Path('artifacts/eval').glob('token_stats_*.json')):
    print('-', path)


## 主实验训练

E1 一定会跑。如果 `RUN_FULL_MAINLINE = True`，会继续跑 E2-E5。


In [ ]:
from pathlib import Path
import gc
import torch
sys.path.insert(0, str(Path.cwd() / 'src'))

from novel_continuation.training import load_training_config, train_baseline_model, train_retrieval_model

main_experiments = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml', 'artifacts/e1_plain_full'),
]

if RUN_FULL_MAINLINE:
    main_experiments.extend([
        ('E2', 'configs/e2_distilgpt2_structured_full.yaml', 'artifacts/e2_structured_full'),
        ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml', 'artifacts/e3_long_context'),
        ('E4', 'configs/e4_distilgpt2_structured_lora.yaml', 'artifacts/e4_lora'),
        ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml', 'artifacts/e5_aux_ranking'),
    ])

for experiment_id, config_path, _model_dir in main_experiments:
    print(f'\n===== 开始训练 {experiment_id} =====')
    config = load_training_config(Path(config_path))
    if config.get('use_retrieval', False):
        trainer, tokenizer = train_retrieval_model(config)
    else:
        trainer, tokenizer = train_baseline_model(config)

    # 训练完立刻释放 GPU 显存，给后续评估和下一轮训练腾空间
    del trainer, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_mb = torch.cuda.mem_get_info()[0] / 1024 / 1024
        print(f'{experiment_id} 训练完成，GPU 显存已释放（剩余 {free_mb:.0f} MB）')
    else:
        print(f'{experiment_id} 训练完成。')

print('\n主实验训练完成。')

## 样本生成、自动评估与人评导出

这一步会为每个已经训练完成的主实验生成：

- `generated_samples_*.jsonl`
- `metrics_*.csv`
- `human_eval_*.csv`


In [ ]:
from novel_continuation.training import load_trained_model_and_tokenizer
from novel_continuation.evaluation import evaluate_generated_rows, write_metrics_csv

# 导入生成和人评相关函数
sys.path.insert(0, str(Path.cwd() / 'scripts'))
from generate_samples import generate_rows, write_jsonl, load_jsonl
from prepare_human_eval import build_human_eval_rows, write_human_eval_csv

test_rows = load_jsonl(Path('data/processed/test.jsonl'))

for experiment_id, _config_path, model_dir in main_experiments:
    sample_path = Path('artifacts/eval') / f'generated_samples_{experiment_id.lower()}.jsonl'
    metrics_path = Path('artifacts/eval') / f'metrics_{experiment_id.lower()}.csv'
    human_eval_path = Path('artifacts/eval') / f'human_eval_{experiment_id.lower()}.csv'

    # ---- 生成样本（带进度条） ----
    print(f'\n----- {experiment_id}: 生成样本 -----')
    generated = generate_rows(
        rows=test_rows,
        model_dir=Path(model_dir),
        max_new_tokens=80,
        use_retrieval=False,
    )
    write_jsonl(generated, sample_path)

    # 释放生成模型显存
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- 自动评估（带进度条） ----
    print(f'----- {experiment_id}: 自动评估 -----')
    model, tokenizer = load_trained_model_and_tokenizer(Path(model_dir))
    metrics = evaluate_generated_rows(generated, model=model, tokenizer=tokenizer)
    write_metrics_csv(metrics, metrics_path)
    print(f'{experiment_id} 指标: ' + ', '.join(f'{k}={v:.4f}' for k, v in metrics.items()))

    # 释放评估模型显存
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- 人评导出 ----
    print(f'----- {experiment_id}: 人评导出 -----')
    human_rows = build_human_eval_rows(generated, system_label=experiment_id)
    write_human_eval_csv(human_rows, human_eval_path)
    print(f'{experiment_id} 评估全部完成。')

print('\n评估文件已生成。')
for path in sorted(Path('artifacts/eval').glob('*')):
    print('-', path)

## Appendix：retrieval 附加实验

只有在 `RUN_APPENDIX_RETRIEVAL = True` 时才会执行这一块。


In [ ]:
if RUN_APPENDIX_RETRIEVAL:
    run(f'{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml')
    run(f'{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl')
    run(f'{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv')
    run(f'{sys.executable} scripts/prepare_human_eval.py --input-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/human_eval_retrieval.csv --system-label Retrieval')
    print('retrieval 附加实验已完成。')
else:
    print('已跳过 retrieval 附加实验。')


## 打包并下载结果

如果 `DOWNLOAD_RESULTS_ZIP = True`，会：

1. 清理中间 checkpoint（节省空间）
2. 只打包 eval 结果 + 训练配置（几 MB），不打包模型权重（几 GB）

In [ ]:
import glob

if DOWNLOAD_RESULTS_ZIP:
    # 1. 清理中间 checkpoint 目录（节省 Colab 磁盘空间）
    for ckpt_dir in sorted(Path('artifacts').rglob('checkpoint-*')):
        if ckpt_dir.is_dir():
            shutil.rmtree(ckpt_dir)
            print(f'已清理: {ckpt_dir}')

    # 2. 收集需要下载的文件（eval 结果 + 每个实验的训练配置）
    download_dir = Path('artifacts/download_package')
    download_dir.mkdir(parents=True, exist_ok=True)

    # 复制 eval 目录（metrics、生成样本、人评 CSV、token stats）
    eval_src = Path('artifacts/eval')
    if eval_src.exists():
        shutil.copytree(eval_src, download_dir / 'eval', dirs_exist_ok=True)

    # 复制每个实验的 training_config.json（记录超参数，写报告用）
    for config_file in Path('artifacts').glob('*/training_config.json'):
        dest = download_dir / config_file.parent.name / config_file.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_file, dest)

    # 3. 打包并下载
    archive_path = shutil.make_archive(
        str(Path('artifacts') / 'colab_results'), 'zip',
        root_dir=str(download_dir)
    )
    size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f'压缩包已生成: {archive_path} ({size_mb:.1f} MB)')

    # 清理临时目录
    shutil.rmtree(download_dir)

    from google.colab import files
    files.download(archive_path)
else:
    print('已跳过结果压缩包下载。')